# Bundle Adjustment — API Demo

How to apply bundle adjustment (BA) to a feedforward pointcloud result.

**Prerequisites:** [Keyframe Extraction](../01_preprocessing/keyframe_extraction.ipynb) → [Feedforward Methods](feedforward_methods.ipynb)

**Not an evaluation.** For benchmark results across `baseline / ba / lc`, see `eval_7scenes_gt.ipynb`. Compute lives in `evals/eval_gt.py`.

Two API styles are demonstrated:

- **Section A**: full pipeline — `creator.run(IMAGES)` → `ba.refine(result)` → `creator.reproject(refined)` → `creator.build_colmap(RECON)`
- **Section B**: cached result path — load `FeedforwardResult` from zarr, call `ba.refine()` without re-running GPU inference (no reproject — `raw_outputs` not persisted)

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import shutil
from pathlib import Path

import torch
import zarr

from collab_splats.pointcloud import BundleAdjustment, BundleAdjustmentConfig
from collab_splats.pointcloud.feedforward import VGGTXCreator
from collab_splats.pointcloud.feedforward.base import FeedforwardResult

## §0 — Setup

Source the common tutorial configuration, then assert that prerequisite notebooks have run.
All paths derive from `CACHE_DIR` set by `tutorial_config.py`.

In [ ]:
%run ../tutorial_config.py

# ── Configuration ─────────────────────────────────────────────────────────────
METHOD    = "vggtx"    # "vggtx" | "mapanything"
VARIANT   = "ba"       # "ba" | "lc" | "" (empty = raw baseline)

VGGTX_ZARR = CACHE_DIR / "vggtx" / "reconstruction.zarr"
RECON      = CACHE_DIR / METHOD / VARIANT if VARIANT else CACHE_DIR / METHOD
RECON.mkdir(parents=True, exist_ok=True)

In [4]:
assert IMAGES.exists() and any(IMAGES.iterdir()), (
    f"No images in {IMAGES}. Run 01_preprocessing/keyframe_extraction first."
)
assert VGGTX_ZARR.exists(), (
    f"Feedforward zarr not found at {VGGTX_ZARR}. Run 02_pointcloud/feedforward_methods first."
)

## §1 — Section A: Full Pipeline

`creator.run()` runs the feedforward model and returns a `FeedforwardResult`.
`BundleAdjustment.refine()` refines poses in-place (returns a new `FeedforwardResult` with updated extrinsics/intrinsics).
`creator.reproject()` re-extracts world points using the refined poses.
`creator.build_colmap()` exports the result to COLMAP format.

In [ ]:
creator = VGGTXCreator()
ba = BundleAdjustment(config=BundleAdjustmentConfig())

# Run feedforward inference; loads model + builds FeedforwardResult with raw_outputs
ff_result = creator.run(IMAGES)
print(f"feedforward: {ff_result.pts3d.shape[0]:,} points, {ff_result.extrinsics.shape[0]} cameras")

# Refine camera poses — returns new FeedforwardResult with updated extrinsics/intrinsics
refined = ba.refine(ff_result)

# Re-project world points using refined poses (requires raw_outputs from creator.run())
reprojected = creator.reproject(refined)
print(f"after BA: {reprojected.pts3d.shape[0]:,} points")

# Export to COLMAP format
colmap_result = creator.build_colmap(RECON)
print(f"COLMAP: {len(colmap_result.points):,} points, {colmap_result.camera_poses.shape[0]} cameras")

## §2 — Section B: Cached Result Path (no GPU re-run)

Load a pre-computed `FeedforwardResult` from the zarr written by `feedforward_methods.ipynb`,
then call `ba.refine()` without re-loading the model.

> **Note:** `creator.reproject()` is not available in this path — `raw_outputs` from the
> original inference run is not persisted to zarr. `pts3d` in the refined result reflects
> the original feedforward geometry, not re-projected points.

In [ ]:
# Load cached feedforward result; restore images from zarr store (needed for track extraction)
ff_result = FeedforwardResult.load_zarr(VGGTX_ZARR)
_store = zarr.open(str(VGGTX_ZARR), mode="r")
ff_result.images = torch.from_numpy(_store["images"][:])   # (N, 3, H, W)

# Refine poses — no model load; uses cached result directly
ba = BundleAdjustment(config=BundleAdjustmentConfig())
refined = ba.refine(ff_result)
print(f"refined: {refined.extrinsics.shape[0]} cameras  pts3d: {refined.pts3d.shape[0]:,} points")

## Where to go next

- **Benchmarks across baseline/BA/LC** — `evals/eval_gt.py` (compute) + `docs/pointcloud/eval_7scenes_gt.ipynb` (viz)
- **Loop closure path** — `collab_splats.pointcloud.LoopClosure`
- **Tunable knobs** — `BundleAdjustmentConfig` (`max_reproj_error`, `lm_steps`, `shared_camera`, `min_inliers_per_frame`, `device`)
- **Source** — `collab_splats/pointcloud/bundle_adjustment.py` (BundleAdjustment class, BundleAdjustmentConfig), `collab_splats/pointcloud/feedforward/base.py` (BaseFeedforwardCreator.run, .reproject)